In [1]:
import sys
import os

# Detect if running on Google Colab
try:
    from google.colab import drive

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    BASE_PATH = "/content/drive/MyDrive/Thesis_Final/fake-new-detection"
else:
    # Local path - go up one level from notebooks to project root
    BASE_PATH = os.path.dirname(os.getcwd())

import sys, os, math, random, json, warnings, datetime
import h5py

# MLflow for experiment tracking

import mlflow
import mlflow.pytorch

# Add BASE_PATH to sys.path so we can import src module

sys.path.insert(0, BASE_PATH)

import numpy as np


## 1. Setup & Dependencies


In [2]:
## 4. Model Initialization (ResNet50-compatible COOLANT)

import src.models.coolant_official as _co_mod
from src.models.base import FastCNN
from pathlib import Path

IMAGE_DIM = 2048  # ResNet50 output dim
TEXT_EMBED = 768  # BERT hidden size

CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,  # 16+16+64
    "h_dim": 64,
    "lr": 1e-3,
    "l2": 0.0,
    "num_epochs": 30,
    "seed": SEED,
    "device": str(DEVICE),
    "save_dir": "./training/checkpoints",
}


# ── Concrete subclass: satisfies the three abstract methods ───────────────────
class ResNetCOOLANT(_co_mod.COOLANT_Official):
    """COOLANT_Official concrete wrapper for ResNet50 2048-dim image features."""

    def encode_text(self, text):
        dummy_img = torch.zeros(text.size(0), 512, device=text.device)
        t, _ = self.similarity_module.encoding(text, dummy_img)
        return t

    def encode_image(self, image):
        dummy_txt = torch.zeros(image.size(0), 512, TEXT_EMBED, device=image.device)
        _, i = self.similarity_module.encoding(dummy_txt, image)
        return i

    def fuse_modalities(self, text_f, image_f):
        return torch.cat([text_f, image_f], dim=-1)


model = ResNetCOOLANT(CONFIG)


# ── Patch 1: EncodingPart.shared_image  512 → IMAGE_DIM ──────────────────────
def _patch_enc(enc):
    layers, done = [], False
    for l in enc.shared_image:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(IMAGE_DIM, l.out_features))
            done = True
        else:
            layers.append(l)
    enc.shared_image = nn.Sequential(*layers)


_patch_enc(model.similarity_module.encoding)
_patch_enc(model.detection_module.encoding)
_patch_enc(model.detection_module.ambiguity_module.encoding)

# ── Patch 2: CLIP.image_projection  512 → IMAGE_DIM ─────────────────────────
layers, done = [], False
for l in model.clip_module.image_projection:
    if isinstance(l, nn.Linear) and not done:
        layers.append(nn.Linear(IMAGE_DIM, l.out_features))
        done = True
    else:
        layers.append(l)
model.clip_module.image_projection = nn.Sequential(*layers)


# ── Patch 3: FastCNN conv input  200 → TEXT_EMBED ────────────────────────────
def _patch_cnn(m):
    m.fast_cnn = FastCNN(
        input_dim=TEXT_EMBED, channel=32, kernel_size=(1, 2, 4, 8)
    ).fast_cnn


_patch_cnn(model.similarity_module.encoding.shared_text_encoding)
_patch_cnn(model.detection_module.encoding.shared_text_encoding)
_patch_cnn(model.detection_module.ambiguity_module.encoding.shared_text_encoding)

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print("Model ready ✓")

ModuleNotFoundError: No module named 'src.models'

## 2. Load Preprocessed Data


In [ ]:
def make_sim_pairs(text, image, label):
    """Paired/unpaired samples from real-news rows for similarity learning."""
    idx = (label == 0).nonzero(as_tuple=True)[0].tolist() or list(range(2))
    t = text[idx].clone()
    i_match = image[idx].clone()
    i_unmatch = image[idx].clone().roll(3, 0)
    return t, i_match, i_unmatch


def soft_xe(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, 1)).sum() / logits.size(0)


def run_epoch(loader, train=True, epoch_num=None):
    if train:
        model.similarity_module.train()
        model.clip_module.train()
        model.detection_module.train()
    else:
        model.eval()

    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    all_y, all_p = [], []

    # Track individual loss components for MLflow
    epoch_sim_loss, epoch_clip_loss, epoch_det_loss = 0.0, 0.0, 0.0
    num_batches = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for text, image, label in loader:
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            label = label.to(DEVICE)
            bs = label.size(0)

            # ── Task 1a: Cosine Similarity ────────────────────────────────
            ft, fi_m, fi_u = make_sim_pairs(text, image, label)
            ta_m, ia_m, _ = model.similarity_module(ft, fi_m)
            ta_u, ia_u, _ = model.similarity_module(ft, fi_u)
            t_cat = torch.cat([ta_m, ta_u])
            i_cat = torch.cat([ia_m, ia_u])
            y_cos = torch.cat(
                [
                    torch.ones(ta_m.size(0), device=DEVICE),
                    -torch.ones(ta_u.size(0), device=DEVICE),
                ]
            )
            ls = loss_cos(t_cat, i_cat, y_cos)
            epoch_sim_loss += ls.item()
            if train:
                optim_sim.zero_grad()
                ls.backward()
                optim_sim.step()

            # ── Task 1b: CLIP Contrastive ─────────────────────────────────
            text_1d = text.mean(1)  # (B, 768) mean-pool over tokens
            ie, te = model.clip_module(image, text_1d)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)

            ts, is_, _ = model.similarity_module(text, image)
            soft_m = is_ @ ts.T * math.exp(0.07)

            lc = (loss_ce(logits, ids) + loss_ce(logits.T, ids)) / 2
            ls2 = (
                soft_xe(logits, F.softmax(soft_m, 1))
                + soft_xe(logits.T, F.softmax(soft_m.T, 1))
            ) / 2
            l_clip = lc + 0.2 * ls2
            epoch_clip_loss += l_clip.item()
            if train:
                optim_clip.zero_grad()
                l_clip.backward()
                optim_clip.step()

            # ── Task 2: Detection ─────────────────────────────────────────
            with torch.no_grad() if not train else torch.enable_grad():
                ie2, te2 = model.clip_module(image, text_1d)
            det, attn, skl = model.detection_module(text, image, te2, ie2)
            ld = loss_ce(det, label) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )
            epoch_det_loss += ld.item()
            if train:
                optim_det.zero_grad()
                ld.backward()
                optim_det.step()

            # ── Metrics ───────────────────────────────────────────────────
            pred = det.argmax(1)
            tot_ok += pred.eq(label).sum().item()
            tot_n += bs
            tot_loss += ld.item() * bs
            all_y.extend(label.cpu().numpy())
            all_p.extend(pred.cpu().numpy())
            num_batches += 1

    # Log batch-level loss components to MLflow (average per epoch)
    if epoch_num is not None:
        prefix = "train" if train else "val"
        mlflow.log_metrics(
            {
                f"{prefix}_sim_loss": epoch_sim_loss / num_batches,
                f"{prefix}_clip_loss": epoch_clip_loss / num_batches,
                f"{prefix}_det_loss": epoch_det_loss / num_batches,
            },
            step=epoch_num,
        )

    return tot_loss / tot_n, tot_ok / tot_n, all_y, all_p

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True, epoch_num=epoch)
    vl_loss, vl_acc, vl_y, vl_p = run_epoch(val_loader, train=False, epoch_num=epoch)
    sch_sim.step()
    sch_clip.step()
    sch_det.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    # Log epoch metrics to MLflow
    mlflow.log_metrics(
        {
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": vl_loss,
            "val_acc": vl_acc,
            "lr_sim": sch_sim.get_last_lr()[0],
            "lr_clip": sch_clip.get_last_lr()[0],
            "lr_det": sch_det.get_last_lr()[0],
        },
        step=epoch,
    )

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
        f"train loss={tr_loss:.4f} acc={tr_acc:.4f}  |  "
        f"val loss={vl_loss:.4f} acc={vl_acc:.4f}"
    )

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        print(f"  ✓ Best val_acc={best_val_acc:.4f} saved.")
        # Log best model checkpoint to MLflow
        mlflow.log_artifact(SAVE_DIR / "best_model.pth", artifact_path="checkpoints")
        mlflow.log_metric("best_val_acc", best_val_acc, step=epoch)

# Log training history as JSON artifact
with open(SAVE_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
mlflow.log_artifact(SAVE_DIR / "training_history.json", artifact_path="metrics")

print(f"\nTraining done. Best val acc: {best_val_acc:.4f}")
print(f"MLflow run ID: {mlflow.active_run().info.run_id}")

[WARNING] Single class detected. Randomly marking 30% as class 1.
After synthesis: {np.int64(0): np.int64(3370), np.int64(1): np.int64(1444)}


In [13]:
## 3. Create HDF5 DataLoaders (Memory-Efficient)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Device setup
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {DEVICE}")

SEED = 42
BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Create dataloaders using HDF5 (loads batches on-demand, no full dataset in RAM)
train_loader, val_loader, test_loader = create_hdf5_dataloaders(
    HDF5_PATH, batch_size=BATCH_SIZE, num_workers=0  # Must be 0 for HDF5
)

print(f"\nDataLoaders ready ✓")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

Device: mps
HDF5Dataset [train]: 3369 samples
HDF5Dataset [val]: 722 samples
HDF5Dataset [test]: 723 samples

DataLoaders created:
  Train: 106 batches
  Val:   23 batches
  Test:  23 batches

DataLoaders ready ✓
  Train batches: 106
  Val batches:   23
  Test batches:  23


## 4. Model Initialization (ResNet50-compatible COOLANT)


In [16]:
import src.models.coolant_official as _co_mod
from src.models.base import FastCNN
from pathlib import Path

IMAGE_DIM = 2048  # ResNet50 output dim
TEXT_EMBED = 768  # BERT hidden size

CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,  # 16+16+64
    "h_dim": 64,
    "lr": 1e-3,
    "l2": 0.0,
    "num_epochs": 30,
    "seed": SEED,
    "device": str(DEVICE),
    "save_dir": "./training/checkpoints",
}


# ── Concrete subclass: satisfies the three abstract methods ───────────────────
class ResNetCOOLANT(_co_mod.COOLANT_Official):
    """COOLANT_Official concrete wrapper for ResNet50 2048-dim image features."""

    def encode_text(self, text):
        dummy_img = torch.zeros(text.size(0), 512, device=text.device)
        t, _ = self.similarity_module.encoding(text, dummy_img)
        return t

    def encode_image(self, image):
        dummy_txt = torch.zeros(image.size(0), 512, TEXT_EMBED, device=image.device)
        _, i = self.similarity_module.encoding(dummy_txt, image)
        return i

    def fuse_modalities(self, text_f, image_f):
        return torch.cat([text_f, image_f], dim=-1)


model = ResNetCOOLANT(CONFIG)


# ── Patch 1: EncodingPart.shared_image  512 → IMAGE_DIM ──────────────────────
def _patch_enc(enc):
    layers, done = [], False
    for l in enc.shared_image:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(IMAGE_DIM, l.out_features))
            done = True
        else:
            layers.append(l)
    enc.shared_image = nn.Sequential(*layers)


_patch_enc(model.similarity_module.encoding)
_patch_enc(model.detection_module.encoding)
_patch_enc(model.detection_module.ambiguity_module.encoding)

# ── Patch 2: CLIP.image_projection  512 → IMAGE_DIM ─────────────────────────
layers, done = [], False
for l in model.clip_module.image_projection:
    if isinstance(l, nn.Linear) and not done:
        layers.append(nn.Linear(IMAGE_DIM, l.out_features))
        done = True
    else:
        layers.append(l)
model.clip_module.image_projection = nn.Sequential(*layers)


# ── Patch 3: FastCNN conv input  200 → TEXT_EMBED ────────────────────────────
def _patch_cnn(m):
    m.fast_cnn = FastCNN(
        input_dim=TEXT_EMBED, channel=32, kernel_size=(1, 2, 4, 8)
    ).fast_cnn


_patch_cnn(model.similarity_module.encoding.shared_text_encoding)
_patch_cnn(model.detection_module.encoding.shared_text_encoding)
_patch_cnn(model.detection_module.ambiguity_module.encoding.shared_text_encoding)

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print("Model ready ✓")

Trainable parameters: 5,610,288
Model ready ✓


## 5. Training Configuration


In [17]:
optim_sim = torch.optim.Adam(
    model.similarity_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["l2"]
)
optim_clip = torch.optim.AdamW(
    model.clip_module.parameters(), lr=1e-3, weight_decay=5e-4
)
optim_det = torch.optim.Adam(
    model.detection_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["l2"]
)

sch_sim = torch.optim.lr_scheduler.StepLR(optim_sim, step_size=10, gamma=0.5)
sch_clip = torch.optim.lr_scheduler.StepLR(optim_clip, step_size=10, gamma=0.5)
sch_det = torch.optim.lr_scheduler.StepLR(optim_det, step_size=10, gamma=0.5)

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce = nn.CrossEntropyLoss()
loss_kl = nn.KLDivLoss(reduction="batchmean")

NUM_EPOCHS = CONFIG["num_epochs"]
SAVE_DIR = Path(CONFIG["save_dir"])
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Optimizers ready ✓")

Optimizers ready ✓


In [ ]:
## 4.5. MLflow Experiment Setup

import mlflow
import mlflow.pytorch

# Initialize MLflow experiment
MLFLOW_EXPERIMENT_NAME = "coolant-vietnamese-fake-news"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# Start MLflow run with descriptive name
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
mlflow_run_name = f"coolant-resnet50-{timestamp}"
mlflow.start_run(run_name=mlflow_run_name)

# Log all CONFIG parameters
mlflow.log_params(CONFIG)
mlflow.log_params(
    {
        "image_dim": IMAGE_DIM,
        "text_embed": TEXT_EMBED,
        "train_samples": len(train_loader.dataset),
        "val_samples": len(val_loader.dataset),
        "test_samples": len(test_loader.dataset),
        "trainable_params": n_params,
    }
)
print(f"MLflow run started: {mlflow_run_name}")
print(f"MLflow UI: http://localhost:5000 (run 'mlflow ui' to view)")

## 6. Helper Functions


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True, epoch_num=epoch)
    vl_loss, vl_acc, vl_y, vl_p = run_epoch(val_loader, train=False, epoch_num=epoch)
    sch_sim.step()
    sch_clip.step()
    sch_det.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    # Log epoch metrics to MLflow
    mlflow.log_metrics(
        {
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "val_loss": vl_loss,
            "val_acc": vl_acc,
            "lr_sim": sch_sim.get_last_lr()[0],
            "lr_clip": sch_clip.get_last_lr()[0],
            "lr_det": sch_det.get_last_lr()[0],
        },
        step=epoch,
    )

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
        f"train loss={tr_loss:.4f} acc={tr_acc:.4f}  |  "
        f"val loss={vl_loss:.4f} acc={vl_acc:.4f}"
    )

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        print(f"  ✓ Best val_acc={best_val_acc:.4f} saved.")
        # Log best model checkpoint to MLflow
        mlflow.log_artifact(SAVE_DIR / "best_model.pth", artifact_path="checkpoints")
        mlflow.log_metric("best_val_acc", best_val_acc, step=epoch)

# Log training history as JSON artifact
with open(SAVE_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
mlflow.log_artifact(SAVE_DIR / "training_history.json", artifact_path="metrics")

print(f"\nTraining done. Best val acc: {best_val_acc:.4f}")
print(f"MLflow run ID: {mlflow.active_run().info.run_id}")

## 7. Training Loop


In [19]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True)
    vl_loss, vl_acc, vl_y, vl_p = run_epoch(val_loader, train=False)
    sch_sim.step()
    sch_clip.step()
    sch_det.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
        f"train loss={tr_loss:.4f} acc={tr_acc:.4f}  |  "
        f"val loss={vl_loss:.4f} acc={vl_acc:.4f}"
    )

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        print(f"  ✓ Best val_acc={best_val_acc:.4f} saved.")

print(f"\nTraining done. Best val acc: {best_val_acc:.4f}")

KeyboardInterrupt: 

## 8. Save Training History


In [ ]:
with open(SAVE_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("History saved.")

## 9. Final Evaluation on Test Set


In [ ]:
model.load_state_dict(torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE))
te_loss, te_acc, te_y, te_p = run_epoch(test_loader, train=False)

pre, rec, f1, _ = precision_recall_fscore_support(
    te_y, te_p, average="weighted", zero_division=0
)

print(f"\n=== Test Results ===")
print(f"Loss      : {te_loss:.4f}")
print(f"Accuracy  : {te_acc:.4f}")
print(f"Precision : {pre:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(
    "\n"
    + classification_report(te_y, te_p, target_names=["Real", "Fake"], zero_division=0)
)

results = {
    "test_loss": te_loss,
    "test_accuracy": te_acc,
    "precision": float(pre),
    "recall": float(rec),
    "f1": float(f1),
}
with open(SAVE_DIR / "test_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Test results saved.")

## 10. Visualizations


In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("COOLANT Training Curves", fontsize=14, fontweight="bold")

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["val_loss"], label="Val")
axes[0].set(title="Detection Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], label="Train")
axes[1].plot(epochs, history["val_acc"], label="Val")
axes[1].set(title="Detection Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
cm = confusion_matrix(te_y, te_p)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=ax,
)
ax.set(title="Confusion Matrix — Test Set", xlabel="Predicted", ylabel="Actual")
plt.tight_layout()
plt.savefig(SAVE_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## 11. Save Final Model


In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": CONFIG,
        "best_val_acc": best_val_acc,
        "test_results": results,
        "history": history,
    },
    SAVE_DIR / "coolant_resnet50_final.pth",
)
print(f"Saved → {SAVE_DIR}/coolant_resnet50_final.pth")
print(f"  Best val acc : {best_val_acc:.4f}")
print(f"  Test acc     : {te_acc:.4f}")
print(f"  Test F1      : {f1:.4f}")